## Fase #4: Modelado / Modeling
1. Variable objetivo y algoritmo seleccionado

La variable objetivo seleccionada es agua_inyectada_m3, que representa el volumen de agua inyectada en las operaciones de fractura hidráulica, expresado en metros cúbicos. Se plantea un problema de regresión supervisada, ya que se busca predecir un valor numérico continuo. Esta elección permite obtener estimaciones cuantitativas útiles para la planificación logística y la previsión de los requerimientos de agua.

Se utiliza el algoritmo RandomForestRegressor, perteneciente a la familia de métodos de aprendizaje supervisado basados en conjuntos de árboles de decisión. Este algoritmo combina las predicciones de múltiples árboles y permite modelar relaciones no lineales e interacciones entre las variables de entrada, sin requerir que se establezca previamente una ecuación matemática que describa esas relaciones.

Como variables predictoras se utilizan la longitud de la rama horizontal, la cantidad de fracturas, la formación productiva, el subtipo de reservorio y el tipo de terminación. Se combinan variables numéricas y categóricas para representar distintas características geológicas y operativas que podrían estar relacionadas con el volumen de agua inyectada.

Además, se implementa un DummyRegressor con estrategia median, que predice la mediana del volumen de agua inyectada del conjunto de entrenamiento. Este modelo funciona como referencia para determinar, durante la fase de evaluación, si el algoritmo principal logra mejorar las predicciones de una estrategia básica.

2. Implementación del modelo

La implementación se realiza en Python mediante la biblioteca Scikit-learn. Se construye un flujo de procesamiento que contempla el tratamiento de valores faltantes y la transformación de las variables categóricas mediante One-Hot Encoding. Las variables numéricas se imputan utilizando la mediana y las categóricas mediante su categoría más frecuente.

Las transformaciones y el algoritmo de regresión se integran en un Pipeline, de modo que el procesamiento se ajuste con los datos de entrenamiento y luego se aplique al conjunto de prueba. El modelo Random Forest se configura inicialmente con 400 árboles, un mínimo de tres observaciones por hoja, una semilla aleatoria fija y procesamiento paralelo.

3. Estrategia de entrenamiento y prueba

Se utiliza una partición temporal con fecha de corte el 1 de enero de 2025. El conjunto de entrenamiento contiene 1.999 operaciones anteriores a 2025, mientras que el conjunto de prueba comprende 623 operaciones correspondientes a 2025–2026. Además, se excluyen del entrenamiento cuatro pozos que también aparecen en el conjunto de prueba.

Esta estrategia busca evaluar la capacidad del modelo para realizar predicciones sobre operaciones posteriores y pozos que no estuvieron presentes durante el entrenamiento. A diferencia de una división aleatoria, la separación temporal respeta el orden cronológico de los datos y permite analizar el desempeño en un período posterior al utilizado para aprender.

Finalmente, ambos modelos se entrenan con los mismos datos y generan predicciones sobre el mismo conjunto de prueba. Se exportan los modelos, las predicciones, los valores observados y los metadatos necesarios para realizar la evaluación en la siguiente fase de CRISP-DM.

In [1]:
%pip install scikit-learn

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------- ----------- 5.8/8.2 MB 33.6 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 24.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# @title Imports

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from comun import leer_dataframe_con_fechas


In [3]:
# @title Preparación de variables

objetivo = "agua_inyectada_m3"
entradas_numericas = ["longitud_rama_horizontal_m", "cantidad_fracturas"]
entradas_categoricas = ["formacion_productiva", "subtipo_reservorio", "tipo_terminacion"]
entradas = entradas_numericas + entradas_categoricas


corte_temporal = pd.Timestamp("2025-01-01")
df_datos_energia = leer_dataframe_con_fechas()
datos_modelo = df_datos_energia[[*entradas, objetivo, "idpozo", "fecha_inicio_fractura", "id_base_fractura_adjiv"]].copy()
train = datos_modelo.loc[datos_modelo["fecha_inicio_fractura"] < corte_temporal].copy()
test = datos_modelo.loc[datos_modelo["fecha_inicio_fractura"] >= corte_temporal].copy()

pozos_compartidos = set(train["idpozo"]).intersection(test["idpozo"])
train = train.loc[~train["idpozo"].isin(pozos_compartidos)].copy()
if train.empty or test.empty:
	raise ValueError("La partición temporal quedó vacía; revisar cobertura de fechas.")

X_train, y_train = train[entradas], train[objetivo]
X_test, y_test = test[entradas], test[objetivo]


In [4]:
codificador = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
prep = ColumnTransformer(
	[
		("num", SimpleImputer(strategy="median"), entradas_numericas),
		(
			"cat",
			Pipeline(
				[
					("imputer", SimpleImputer(strategy="most_frequent")),
					("onehot", codificador),
				]
			),
			entradas_categoricas,
		),
	]
)

modelo = Pipeline(
	[
		("prep", prep),
		("regresor", RandomForestRegressor(n_estimators=400, min_samples_leaf=3, random_state=42, n_jobs=-1)),
	]
)
referencia = DummyRegressor(strategy="median")

modelo.fit(X_train, y_train)
referencia.fit(X_train, y_train)
pred_modelo = modelo.predict(X_test)
pred_referencia = referencia.predict(X_test)

print(f"Entrenamiento: {len(train):,} registros hasta 2024")
print(f"Prueba temporal: {len(test):,} registros de 2025-2026")
print(f"Pozos retirados del entrenamiento por aparecer en prueba: {len(pozos_compartidos)}")


Entrenamiento: 4,273 registros hasta 2024
Prueba temporal: 703 registros de 2025-2026
Pozos retirados del entrenamiento por aparecer en prueba: 6


In [5]:
from comun import exportar_modelos

exportar_modelos(
	modelo=modelo,
	referencia=referencia,
	X_test=X_test,
	y_test=y_test,
	test_metadata=test[
		[
			"id_base_fractura_adjiv",
			"idpozo",
			"fecha_inicio_fractura",
		]
	],
)

print("Modelos y datos de evaluación exportados.")


Modelos y datos de evaluación exportados.


### Interpretación de la partición

El entrenamiento utiliza **1.999 operaciones anteriores a 2025** y la prueba conserva **623 operaciones de 2025–2026**. Se retiraron del entrenamiento cuatro pozos que reaparecen en prueba. Esta separación es más exigente y más cercana a un uso futuro que una división aleatoria, porque el modelo aprende del pasado y se evalúa en un período posterior. También permite detectar cambios de tecnología o de prácticas operativas que una partición aleatoria podría ocultar.